# Tree construction algorithm

Builds full summary tree of given folders (7th prototype). 

### Nesting structure
unit (paragraph/table/figure) -> section -> document -> folder -> ... -> ROOT

### Structure-based parsing 
Used rather than word count-based naive chunking characteristic of RAPTOR.
Converts files to Markdown and uses Markdown headings; tables are self-contained units; figured captioned by Gemma3:27b; logos + page numbers skipped; paragraphs/tables stitched across page breaks.

### Models
1. gpt-oss:120b for text
2. gemma3:27b for images and figures (gpt-oss is text-only)

### Improvement from prototype 6
The previous version traversed each document sequentially, interleaving text and image calls. As a result, Ollama kept evicting and reloading the 120B between calls (even a single reload could take minutes), resulting in significant delays. 
This was solved by implementing summaries in three phases:
1. AlL figures described by gemma3:27b in a row
2. Then, all text chunks consecutively summarised with gpt-oss:120b
3. Finally, these summaries are combined in their original ordering from the documents to build sections -> documents -> folders -> ROOT hierarchy 

Therefore, this only needs 2 model swaps instead of hundreds. Program is crash-resumable and also includes a progress bar showing the current phase with a live ETA

### Instructions to Run

ssh -N -L 11528:172.17.0.1:11434 asharma@ollama.res.oicr.on.ca
curl http://localhost:11528/api/tags

export OLLAMA_HOST=localhost:11528
ollama pull gpt-oss:120b      
ollama pull gemma3:27b         
ollama pull nomic-embed-text  

pip install -r requirements.txt
jupyter lab prototype7.ipynb

In [ ]:
%pip install ollama pymupdf pymupdf4llm python-docx pandas openpyxl rank-bm25 tqdm numpy # Install necessary packages; redundant if command "pip install -r requirements.txt" was already run

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import importlib, sys, os, json, hashlib, re, time, textwrap, base64, threading # modules
import concurrent.futures as cf # parallel
from pathlib import Path # file paths
from dataclasses import dataclass, field #
from typing import List, Dict, Any, Optional, Tuple

required = {
    "ollama": "ollama", "fitz": "pymupdf", "pymupdf4llm": "pymupdf4llm",
    "docx": "python-docx", "pandas": "pandas", "openpyxl": "openpyxl",
    "rank_bm25": "rank-bm25", "tqdm": "tqdm", "numpy": "numpy",
} # import name → pip install name
missing = [pkg for mod, pkg in required.items() if importlib.util.find_spec(mod) is None] # 
if missing:
    print(f"Missing — run:  pip install {' '.join(missing)}")
    raise SystemExit(1)

import ollama
import fitz                       
import docx as _docx
import pandas as pd
import numpy as np
from rank_bm25 import BM25Okapi
from tqdm import tqdm

#pymupdf4llm converts pdf to markdown for structure-aware parsing
try:
    import pymupdf4llm
    HAVE_PYMUPDF4LLM = True
except Exception as _e:
    HAVE_PYMUPDF4LLM = False
    print(f"pymupdf4llm import failed ({_e}); PDF loader will fall back to a basic parse.")

print("All packages OK" + ("" if HAVE_PYMUPDF4LLM else "  (pymupdf4llm missing)")) # lenient in case pymupdf4llm can't be imported

All packages OK


In [ ]:
# Configs

OLLAMA_URL    = "http://localhost:11528"   # tunnel endpoint
SUMMARY_MODEL = "gpt-oss:120b"             # for text summaries
VISION_MODEL  = "gemma3:27b"               # for figure descriptions (gpt-oss is text-only)
AGENT_MODEL   = "gpt-oss:120b"             # agentic traversal
CHAT_MODEL    = "gpt-oss:120b"             # final answer
EMBED_MODEL   = "nomic-embed-text"         # embed text to vectors for comparative baseline

# defining folders
DOCS_ROOT       = Path("folders")          # root of the document library
CACHE_DIR       = Path("tree_cache")       # so that it saves progress to a cache node-by-node in case the program crashes
TREE_FILE       = CACHE_DIR / "corpus_tree.json"
NODE_CACHE_DIR  = CACHE_DIR / "nodes"      # per-node cache
PARSE_CACHE_DIR = CACHE_DIR / "parse"      # parsed-document cache to skip re-parsing
IMG_DIR         = CACHE_DIR / "images"     # extracted figures per document

# parallelism for speed
NUM_WORKERS     = 4      # concurrent text requests; match server OLLAMA_NUM_PARALLEL
VISION_WORKERS  = 2      # concurrent figure requests given vision model is heavier per call
GEN_NUM_PREDICT = 100000    # max output tokens per summary (bounds per-call latency)
KEEP_ALIVE      = "30m"  # keep the large model resident between calls
DISABLE_THINKING = True  # gpt-oss is a reasoning model; skip visible thinking for speed
SKIP_SUMMARY_WORDS = 8   # text units this short are stored verbatim so no LLM call
RETRY_WAIT_MAX  = 60     # max seconds between retries while waiting for Ollama to reconnect
READY_POLL      = 10     # seconds between startup readiness checks

# images
DESCRIBE_IMAGES = True   # yes describe images; this toggle is probably irrelevant
MD_DPI          = 150    # rasterization resolution
LOGO_MIN_PX     = 100    # images with area < LOGO_MIN_PX**2 are treated as logos
DROP_LOGO_BY_DESC = True # also skip a figure if its description says it is a logo

# printing
VERBOSE_CALLS      = True   # print each call's LLM-generated summary as it completes
CORPUS_PRINT_DEPTH = 3      # for printing the final summary tree
SHOW_LEAVES_IN_TREE = False # don't print leaves bc that would cause excessive output logs

# no force rebuild or reparse so that if the program crashes, it retains the cache and can resume from where the program left off
FORCE_REBUILD = False
FORCE_REPARSE = False

for _d in (CACHE_DIR, NODE_CACHE_DIR, PARSE_CACHE_DIR, IMG_DIR):
    _d.mkdir(parents=True, exist_ok=True)
print(f"Docs root  : {DOCS_ROOT.resolve()}")
print(f"Node cache : {NODE_CACHE_DIR}/   parse cache: {PARSE_CACHE_DIR}/")
print(f"Text model : {SUMMARY_MODEL}  (workers={NUM_WORKERS})")
print(f"Vision     : {VISION_MODEL}  (workers={VISION_WORKERS}, describe={DESCRIBE_IMAGES})")
print(f"Schedule   : phase 1 figures -> phase 2 text -> phase 3 combine (one load per model)")
print(f"Speedups   : parse+node cache, short-skip<={SKIP_SUMMARY_WORDS}w, "
      f"num_predict={GEN_NUM_PREDICT}, keep_alive={KEEP_ALIVE}")

Docs root  : /Users/asharma/Desktop/Project Algorithm/folders
Node cache : tree_cache/nodes/   parse cache: tree_cache/parse/
Text model : gpt-oss:120b  (workers=4)
Vision     : gemma3:27b  (workers=2, describe=True)
Schedule   : phase 1 figures -> phase 2 text -> phase 3 combine (one load per model)
Speedups   : parse+node cache, short-skip<=8w, num_predict=100000, keep_alive=30m


In [ ]:
OLLAMA_TIMEOUT = 300   # seconds per request with buffer bc gpt-oss:120b can be slow to cold-load
client = ollama.Client(host=OLLAMA_URL, timeout=OLLAMA_TIMEOUT) # defines ollama client

# input is client.list(), output is model strings
def _model_names(list_resp) -> List[str]:
    raw = list_resp.get("models", []) if hasattr(list_resp, "get") else getattr(list_resp, "models", []) # gets per-model entries accounting for if it is dict structure
    out = []
    # loops over every model entry
    for m in raw:
        name = getattr(m, "model", None) or getattr(m, "name", None) # looks for relevant attribute by searching "model" and "name"
        # if attribute lookup found nothing, and m is actually a dict, retry the same two keys via dictionary access
        if name is None and isinstance(m, dict):
            name = m.get("model") or m.get("name")
        if name:
            out.append(name)
        
        # if name was found, append to index
    return out

try:
    names = _model_names(client.list())
    print("Connected to Ollama. Available models:")
    # prints all the model names found
    for n in names:
        print(f"  {n}")
    _need = [SUMMARY_MODEL, EMBED_MODEL] + ([VISION_MODEL] if DESCRIBE_IMAGES else [])

    # lists all required models that weren't successfully loaded
    for required_model in _need:
        if not any(required_model in n for n in names):
            print(f"\nWARNING: '{required_model}' not found. Pull it with:")
            print(f"  OLLAMA_HOST=localhost:11528 ollama pull {required_model}")

# exception if Ollama can't be reached
except Exception as e:
    print(f"Cannot reach Ollama at {OLLAMA_URL}: {type(e).__name__}: {e}")
    print("Make sure the SSH tunnel is running:")
    print("  ssh -N -L 11528:172.17.0.1:11434 asharma@ollama.res.oicr.on.ca")

Connected to Ollama. Available models:
  nomic-embed-text:latest
  gemma3:27b
  gpt-oss:120b
  gemma3:270m
  llama3.1:latest
  llama3.1:8b
  medgemma:27b
  gpt-oss:20b
  codellama:latest
  llama3.1:70b
  mistral:latest
  llama3:70b-instruct
  mistral-small3.1:latest

Tip: for real parallelism set OLLAMA_NUM_PARALLEL on the server (e.g. OLLAMA_NUM_PARALLEL=4) and match it with NUM_WORKERS.


In [ ]:
_t0 = time.time()  # record start time so we can measure how long the call takes
try:  # attempt the probe; catch any failure instead of crashing
    _r = client.chat(model=SUMMARY_MODEL,  # send a chat request to the summary model
                     messages=[{"role": "user", "content": "Reply with the single word: ok"}],  # minimal prompt; cheap to run
                     options={"num_predict": 5}, keep_alive=KEEP_ALIVE)  # cap output at 5 tokens; keep model resident after
    print(f"chat() OK in {time.time() - _t0:.1f}s  ->  {_r['message']['content'].strip()!r}")  # report elapsed seconds and the trimmed reply
except Exception as e:  # if the call raised for any reason
    print(f"chat() FAILED after {time.time() - _t0:.1f}s: {type(e).__name__}: {e}")  # report elapsed time, error type name, and message

chat() OK in 0.4s  ->  ''


In [ ]:
@dataclass  # auto-generate __init__, __repr__, __eq__ from the fields below
class TreeNode:  # one node in the summary tree
    """node_type: 'root' | 'folder' | 'document' | 'section' | 'chunk'.
    For 'chunk' nodes, metadata['kind'] is 'text' | 'table' | 'image';
    a skipped logo chunk carries metadata['skipped'] = 'logo'."""  # docstring; explains node_type and chunk metadata
    node_id:   str  # unique id for this node; used as the cache key
    node_type: str  # role in the tree; root, folder, document, section, or chunk
    name:      str  # human-readable label; filename or section title
    path:      str  # filesystem path for folders and documents; empty otherwise
    summary:   str  # the generated summary text for this node
    content:   str = ""  # raw text, table, or image description; only set on leaf chunks
    children:  List["TreeNode"] = field(default_factory=list)  # child nodes; new empty list per instance
    metadata:  Dict[str, Any]   = field(default_factory=dict)  # extra info like kind, page, source_file; new empty dict per instance

    def to_dict(self) -> Dict:  # serialize this node and its subtree to plain dicts for json
        return {"node_id": self.node_id, "node_type": self.node_type, "name": self.name,  # copy the scalar fields
                "path": self.path, "summary": self.summary, "content": self.content,  # copy more scalar fields
                "children": [c.to_dict() for c in self.children], "metadata": self.metadata}  # recurse into children; pass metadata through

    @classmethod  # builds an instance without needing an existing one
    def from_dict(cls, d: Dict) -> "TreeNode":  # rebuild a node from its dict form
        node = cls(node_id=d["node_id"], node_type=d["node_type"], name=d["name"],  # read required fields; raises if missing
                   path=d["path"], summary=d["summary"], content=d.get("content", ""),  # content defaults to empty if absent
                   metadata=d.get("metadata", {}))  # metadata defaults to empty dict if absent
        node.children = [cls.from_dict(c) for c in d.get("children", [])]  # recursively rebuild children; empty if none
        return node  # hand back the reconstructed node

    def is_leaf(self) -> bool:  # true only for leaf units
        return self.node_type == "chunk"  # chunks are the leaves; everything else has children

    def count_nodes(self) -> int:  # total nodes in this subtree including self
        return 1 + sum(c.count_nodes() for c in self.children)  # count self plus every descendant recursively

    def count_leaves(self) -> int:  # total leaf units in this subtree
        if self.is_leaf():  # a chunk counts as one leaf
            return 1  # base case; stop recursing
        return sum(c.count_leaves() for c in self.children)  # otherwise sum leaves across children


def _make_id(path: str, extra: str = "") -> str:  # build a short stable id from a path plus optional suffix
    return hashlib.md5(f"{path}|{extra}".encode()).hexdigest()[:12]  # md5 the joined string; keep first 12 hex chars

print("TreeNode class ready")  # confirm the cell ran

TreeNode class ready


In [ ]:
_THINK = {"use": DISABLE_THINKING}  # a little mutable flag for whether we send think=false, wrapped in a dict so the inner functions can flip it


def _wait_backoff(attempt: int):
    # sleeps a bit longer each retry, growing 5s 10s 20s 40s 80s but capped, so we don't hammer the server while it's down
    time.sleep(min(RETRY_WAIT_MAX, 5 * (2 ** min(attempt - 1, 4))))


def _note_waiting(err, attempt: int):
    # prints a "still waiting" note, but only now and then so the logs don't get spammed on every retry
    if attempt == 1 or attempt % 5 == 0:
        tqdm.write(f"[waiting for Ollama] {type(err).__name__}: {err} — "
                   f"retrying (attempt {attempt}); will resume when it reconnects.")


def _chat(messages: List[Dict], num_predict: Optional[int] = None):
    # the main text call, and the key thing is it never gives up, if Ollama drops it just waits and retries forever rather than crashing or caching junk
    opts = {"temperature": 0, "num_predict": num_predict or GEN_NUM_PREDICT}
    attempt = 0
    while True:
        kw = dict(model=SUMMARY_MODEL, messages=messages, options=opts, keep_alive=KEEP_ALIVE)
        if _THINK["use"]:
            kw["think"] = False
        try:
            return client.chat(**kw)
        except TypeError:
            _THINK["use"] = False
            continue
        except Exception as e:
            if _THINK["use"]:
                _THINK["use"] = False
                continue
            attempt += 1
            _note_waiting(e, attempt)
            _wait_backoff(attempt)


def _words(s: str) -> List[str]:
    # just splits text into words, counting every run of non-whitespace, and returns an empty list if there's nothing
    return re.findall(r"\S+", s or "")


def _enforce_not_longer(summary: str, source: str) -> str:
    # a safety net, a summary should never be wordier than what it summarises, so if it is we just throw it out and keep the source
    sw, mw = len(_words(source)), len(_words(summary))
    if sw and mw > sw:
        return source.strip()
    return summary


def _llm_summarise(text: str, context_hint: str = "") -> str:
    # summarises one leaf, a paragraph or a table, with a word budget tied to the source length, and a placeholder if there's nothing to do
    if not text.strip():
        return "(empty)"
    src_words = len(_words(text))
    cap = max(1, min(200, src_words))
    hint = f" The content comes from: {context_hint}." if context_hint else ""
    prompt = (
        f"Summarise the following text.{hint} "
        "Focus on key topics, concepts, and specific information (names, numbers, "
        "procedures, entities). If it is a Markdown table, state what it tabulates, "
        "its columns, and notable values. "
        f"Your summary MUST be shorter than the source and at most {cap} words; it "
        "need not reach that limit. If the source is very short, return it nearly "
        "verbatim or shorter, never longer. Respond with ONLY the summary.\n\n"
        f"{text}"
    )
    resp = _chat([{"role": "user", "content": prompt}])
    return _enforce_not_longer(resp["message"]["content"].strip(), text)


def _llm_combine_summaries(summaries: List[str], label: str) -> str:
    # folds a bunch of child summaries up into one parent summary, which is how the tree gets built bottom up
    if not summaries:
        return "(no content)"
    joined = "\n".join(f"- {s}" for s in summaries)
    prompt = (
        f"You are summarising a section, document, or folder called '{label}'. Below "
        "are summaries of its contents. Write ONE specific summary of the overall "
        "scope and key topics. It MUST be shorter than the combined input below and "
        "at most 200 words; it need not reach that limit. Respond with ONLY the summary.\n\n"
        f"{joined}"
    )
    resp = _chat([{"role": "user", "content": prompt}])
    return _enforce_not_longer(resp["message"]["content"].strip(), joined)


def _resolve_image(image_path: str) -> Optional[Path]:
    # tracks down an extracted figure on disk, trying the path as given, then the image folder, then a recursive search as a last resort
    if not image_path:
        return None
    p = Path(image_path)
    if p.exists():
        return p
    cand = IMG_DIR / p.name
    if cand.exists():
        return cand
    hits = list(IMG_DIR.rglob(p.name))
    return hits[0] if hits else None


def _llm_describe_image(image_path: str, caption_hint: str = "") -> str:
    # describes a figure with the vision model, and like _chat it waits and retries forever rather than dying if the call fails
    p = _resolve_image(image_path)
    if p is None:
        base = f" Caption: {caption_hint}" if caption_hint else ""
        return f"(figure; image file not found){base}"
    hint = f" The figure's caption is: {caption_hint}." if caption_hint else ""
    prompt = (
        "Describe this figure from a document in 3-5 sentences so it can be found by "
        f"search.{hint} State the kind of visual (chart, diagram, photo, schematic, "
        "logo), what it depicts, any axis labels / units / labelled parts you can read, "
        "and the main takeaway. Respond with ONLY the description."
    )
    attempt = 0
    while True:
        try:
            resp = client.chat(model=VISION_MODEL,
                               messages=[{"role": "user", "content": prompt, "images": [str(p)]}],
                               options={"temperature": 0, "num_predict": GEN_NUM_PREDICT},
                               keep_alive=KEEP_ALIVE)
            return resp["message"]["content"].strip()
        except Exception as e:
            attempt += 1
            _note_waiting(e, attempt)
            _wait_backoff(attempt)


def _llm_embed(text: str) -> Optional[List[float]]:
    # grabs an embedding vector for some text, and since embeddings aren't critical it just returns None on failure instead of blowing up
    try:
        return client.embeddings(model=EMBED_MODEL, prompt=text)["embedding"]
    except Exception:
        return None


print("LLM helpers ready (gpt-oss chat wrapper + summarise/combine/vision/embed)")

LLM helpers ready (gpt-oss chat wrapper + summarise/combine/vision/embed)


In [ ]:
def _node_cache_path(node_id: str) -> Path:
    # just builds the on-disk path for a node's cache file, one json per node id
    return NODE_CACHE_DIR / f"{node_id}.json"


def _load_cached_node(node_id: str) -> Optional["TreeNode"]:
    # loads a node back from cache if it's there, and quietly returns None if the file's missing or unreadable, so a corrupt file just means "recompute it"
    p = _node_cache_path(node_id)
    if not p.exists():
        return None
    try:
        with open(p, encoding="utf-8") as f:
            return TreeNode.from_dict(json.load(f))
    except Exception:
        return None


def _save_cached_node(node: "TreeNode") -> None:
    # writes a node to cache atomically, writing to a temp file first and then renaming, so a crash mid-write can't leave a half-written file behind
    p = _node_cache_path(node.node_id)
    tmp = p.with_suffix(".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(node.to_dict(), f, ensure_ascii=False)
    tmp.replace(p)


def clear_node_cache() -> int:
    # wipes the whole node cache and tells you how many files it deleted, which is what FORCE_REBUILD uses to start fresh
    n = 0
    for p in NODE_CACHE_DIR.glob("*.json"):
        p.unlink(); n += 1
    print(f"Cleared {n} cached node(s) from {NODE_CACHE_DIR}")
    return n


def cache_status() -> None:
    # a quick report of what's in the cache, counting nodes by type and adding up the total size on disk
    counts: Dict[str, int] = {}
    total_bytes = 0
    for p in NODE_CACHE_DIR.glob("*.json"):
        total_bytes += p.stat().st_size
        try:
            with open(p, encoding="utf-8") as f:
                k = json.load(f).get("node_type", "other")
            counts[k] = counts.get(k, 0) + 1
        except Exception:
            counts["other"] = counts.get("other", 0) + 1
    print(f"Per-node cache @ {NODE_CACHE_DIR}:")
    for k, v in sorted(counts.items()):
        print(f"  {k:<10s}: {v}")
    print(f"  total size: {total_bytes / 1024:.1f} KB")

_FAIL_RE = re.compile(  # a pattern that matches the failure placeholders we leave behind when a call couldn't complete
    r"\(summary unavailable|\(image description unavailable|\(error:|\(worker crashed",
    re.I)


def repair_cache() -> int:
    # cleans up after a bad run, it deletes any node whose text is a failure placeholder so it recomputes, and also drops the combine nodes above them so they rebuild from clean children, but it keeps all the good leaf summaries so you don't lose real progress
    poisoned, combine = [], []
    for p in NODE_CACHE_DIR.glob("*.json"):
        try:
            d = json.loads(p.read_text(encoding="utf-8"))
        except Exception:
            poisoned.append(p)
            continue
        text = (d.get("summary", "") or "") + "\n" + (d.get("content", "") or "")
        if _FAIL_RE.search(text):
            poisoned.append(p)
        elif d.get("node_type") in ("section", "document", "folder", "root"):
            combine.append(p)
    for p in poisoned:
        try: p.unlink()
        except Exception: pass
    n_comb = 0
    if poisoned:
        for p in combine:
            try: p.unlink(); n_comb += 1
            except Exception: pass
    if poisoned or n_comb:
        print(f"Repaired cache: removed {len(poisoned)} failed node(s)"
              + (f" + {n_comb} combine node(s) to rebuild" if n_comb else ""))
    else:
        print("Cache clean: no failed (could-not-connect) nodes found")
    return len(poisoned)


print("Per-node cache helpers ready")

Per-node cache helpers ready


In [ ]:
def _clean(text: str) -> str:
    # collapses any run of whitespace down to single spaces and trims the ends, so text is tidy
    return re.sub(r"\s+", " ", text).strip()

_PAGE_ARTIFACT_PATTERNS = [  # patterns for the various ways a page number or footer shows up, so we can spot and drop them
    re.compile(r"^page\s+\d+\s+of\s+\d+$", re.I),
    re.compile(r"^page\s+\d+$", re.I),
    re.compile(r"^\d+\s+of\s+\d+$", re.I),
    re.compile(r"^p\s*a\s*g\s*e\s*\|?\s*\d+$", re.I),
    re.compile(r"^\d+\s*\|\s*page$", re.I),
    re.compile(r"^-\s*\d+\s*-$"),
]


def _is_page_artifact(text: str) -> bool:
    # decides whether a line is just a page number or footer, so it never becomes a real paragraph
    t = text.strip().strip("*_ ").strip()
    if not t:
        return True
    if any(p.match(t) for p in _PAGE_ARTIFACT_PATTERNS):
        return True
    if re.fullmatch(r"\d{1,4}", t):
        return True
    return False


def _is_table_sep(line: str) -> bool:
    # checks if a line is a markdown table separator row, the |---|---| kind, which is how we know a table starts
    s = line.strip()
    if "-" not in s or "|" not in s:
        return False
    return all(ch in "|:- " for ch in s)


_TABLE_CAP = re.compile(r"(?:\*\*)?\s*(?:table|tbl)\s*\.?\s*\d+", re.I)  # matches "Table 3" style captions
_FIG_CAP   = re.compile(r"(?:\*\*)?\s*(?:figure|fig)\s*\.?\s*\d+", re.I)  # matches "Figure 3" style captions


def _trailing_caption(text: str, kind: str = "table"):
    # pulls a table caption off the end of a paragraph, returning the caption and whatever text was left in front of it
    pat = _TABLE_CAP if kind == "table" else _FIG_CAP
    if pat.match(text.strip()):
        return text.strip(), ""
    matches = list(pat.finditer(text))
    if matches:
        start = matches[-1].start()
        return text[start:].strip(), text[:start].strip()
    return None


def _leading_caption(text: str, kind: str = "image"):
    # pulls a figure caption off the start of a paragraph, returning the caption line and the remainder after it
    pat = _FIG_CAP if kind == "image" else _TABLE_CAP
    stripped = text.strip()
    if not pat.match(stripped):
        return None
    parts = stripped.split("\n", 1)
    return parts[0].strip(), (parts[1].strip() if len(parts) > 1 else "")


def _parse_markdown_blocks(md_text: str, page: Optional[int]) -> List[Dict]:
    # walks one page of markdown line by line and turns it into ordered typed blocks, headings, text, tables, and images
    lines = md_text.split("\n")
    blocks: List[Dict] = []
    buf: List[str] = []

    def flush_text():
        # empties the running text buffer into a text block, dropping it if it's empty or just a page number
        if buf:
            t = _clean(" ".join(buf))
            buf.clear()
            if t and not _is_page_artifact(t):
                blocks.append({"type": "text", "text": t, "page": page})

    i, n = 0, len(lines)
    while i < n:
        s = lines[i].strip()
        if not s:
            flush_text(); i += 1; continue

        m = re.match(r"^(#{1,6})\s+(.*)$", s)
        if m:
            flush_text()
            title = _clean(m.group(2))
            if title and not _is_page_artifact(title):
                blocks.append({"type": "heading", "level": len(m.group(1)),
                               "text": title, "page": page})
            i += 1; continue

        im = re.match(r"^!\[(.*?)\]\((.*?)\)\s*$", s)
        if im:
            flush_text()
            blocks.append({"type": "image", "alt": _clean(im.group(1)),
                           "path": im.group(2).strip(), "page": page})
            i += 1; continue

        if s.startswith("|") and i + 1 < n and _is_table_sep(lines[i + 1]):
            flush_text()
            tbl = []
            while i < n and lines[i].strip().startswith("|"):
                tbl.append(lines[i].rstrip())
                i += 1
            blocks.append({"type": "table", "text": "\n".join(tbl), "page": page})
            continue

        buf.append(s)
        i += 1

    flush_text()
    return blocks


def _reattach_captions(blocks: List[Dict]) -> List[Dict]:
    # fixes captions that landed on the wrong block, moving a "Table N" caption onto its table and a "Figure N" caption onto its image
    drop = set()
    for idx, blk in enumerate(blocks):
        if blk["type"] == "table":
            j = idx - 1
            while j >= 0 and j in drop:
                j -= 1
            if j >= 0 and blocks[j]["type"] == "text":
                res = _trailing_caption(blocks[j]["text"], "table")
                if res:
                    caption, remainder = res
                    blk["caption"] = caption
                    blk["text"] = caption + "\n\n" + blk["text"]
                    if remainder:
                        blocks[j]["text"] = remainder
                    else:
                        drop.add(j)
        elif blk["type"] == "image":
            k = idx + 1
            while k < len(blocks) and k in drop:
                k += 1
            if k < len(blocks) and blocks[k]["type"] == "text":
                res = _leading_caption(blocks[k]["text"], "image")
                if res:
                    caption, remainder = res
                    blk["caption"] = caption
                    if remainder:
                        blocks[k]["text"] = remainder
                    else:
                        drop.add(k)
    return [b for i, b in enumerate(blocks) if i not in drop]


def _filter_repeated(blocks: List[Dict], num_pages: int) -> List[Dict]:
    # drops short lines that show up on lots of pages, since those are almost always running headers or footers, not content
    if num_pages < 4:
        return blocks
    seen: Dict[str, set] = {}
    for b in blocks:
        if b["type"] == "text" and len(b["text"].split()) <= 10:
            seen.setdefault(b["text"], set()).add(b.get("page"))
    threshold = max(2, (num_pages + 1) // 2)
    repeated = {t for t, pages in seen.items() if len(pages) >= threshold}
    return [b for b in blocks
            if not (b["type"] == "text" and b["text"] in repeated)]


def _build_section_tree(blocks: List[Dict]) -> List[Dict]:
    # turns the flat list of blocks into a nested section tree using heading levels, with anything before the first heading kept as front matter
    root: List[Dict] = []
    stack = []
    frontmatter = {"title": "(front matter)", "level": 0,
                   "paragraphs": [], "subsections": [], "page": None}

    def add_para(p):
        # drops a paragraph into the current open section, or into front matter if no section is open yet
        (stack[-1][1]["paragraphs"] if stack else frontmatter["paragraphs"]).append(p)

    for blk in blocks:
        if blk["type"] == "heading":
            sec = {"title": blk["text"], "level": blk["level"],
                   "paragraphs": [], "subsections": [], "page": blk.get("page")}
            while stack and stack[-1][0] >= blk["level"]:
                stack.pop()
            (stack[-1][1]["subsections"] if stack else root).append(sec)
            stack.append((blk["level"], sec))
        else:
            add_para(blk)

    sections = []
    if frontmatter["paragraphs"]:
        sections.append(frontmatter)
    sections.extend(root)
    return sections

def _docx_table_to_md(tbl) -> str:
    # converts a docx table into a markdown grid, padding ragged rows and escaping pipes so the structure survives
    rows = []
    for r in tbl.rows:
        rows.append([_clean(c.text) for c in r.cells])
    if not rows:
        return ""
    ncol = max(len(r) for r in rows)
    rows = [r + [""] * (ncol - len(r)) for r in rows]
    def fmt(r):
        # formats one row as a markdown table line, escaping any literal pipes inside cells
        return "| " + " | ".join(c.replace("|", r"\|") for c in r) + " |"
    out = [fmt(rows[0]), "| " + " | ".join(["---"] * ncol) + " |"]
    out += [fmt(r) for r in rows[1:]]
    return "\n".join(out)


def _docx_heading_level(style_name: str) -> int:
    # reads the heading depth out of a docx style name like "Heading 2", defaulting to 1 if there's no number
    m = re.search(r"(\d+)", style_name)
    return int(m.group(1)) if m else 1

def _image_dims(p: Path):
    # returns an image's width and height in pixels, or zeros if it can't be opened, used to spot tiny logo images
    try:
        pix = fitz.Pixmap(str(p))
        return pix.width, pix.height
    except Exception:
        return 0, 0


def _filter_logo_images(blocks: List[Dict]) -> List[Dict]:
    # throws out logos and decorative images, anything tiny or byte-identical across pages, so they aren't mistaken for real figures and the text around them can stitch back together
    imgs = [b for b in blocks if b["type"] == "image"]
    if not imgs:
        return blocks
    by_hash: Dict[str, list] = {}
    for b in imgs:
        p = _resolve_image(b.get("path", ""))
        if p is None:
            b["_logo"] = True
            continue
        try:
            h = hashlib.md5(p.read_bytes()).hexdigest()
        except Exception:
            h = None
        w, ht = _image_dims(p)
        if w and ht and w * ht < LOGO_MIN_PX * LOGO_MIN_PX:
            b["_logo"] = True
        if h:
            by_hash.setdefault(h, []).append(b)
    n_drop = 0
    for grp in by_hash.values():
        if len(grp) >= 2:
            for b in grp:
                b["_logo"] = True
    kept = []
    for b in blocks:
        if b.get("_logo"):
            n_drop += 1
            continue
        b.pop("_logo", None)
        kept.append(b)
    return kept


def _looks_like_logo(desc: str) -> bool:
    # a backup check that flags an image as a logo based on its description mentioning words like logo, letterhead, or branding
    return bool(re.search(r"\blogos?\b|\bletterhead\b|\bbranding\b", desc or "", re.I))


_TERMINAL_TRAIL = "\"'\u2019\u201d)]\u00bb"  # trailing quotes and brackets to peel off before checking sentence endings


def _ends_sentence(a: str) -> bool:
    # decides whether a chunk of text ends on a real sentence boundary, ignoring trailing quotes and brackets
    t = a.rstrip().rstrip(_TERMINAL_TRAIL)
    return bool(t) and t[-1] in ".!?"


def _is_link_only(s: str) -> bool:
    # checks if a line is nothing but a link, a markdown link or a bare url, which usually belongs with the paragraph before it
    s = s.strip()
    return bool(re.fullmatch(r"\[.*?\]\(.*?\)", s)
                or re.fullmatch(r"<?https?://\S+>?", s)
                or re.fullmatch(r"www\.\S+", s))


def _join_text(a: str, b: str) -> str:
    # glues two text pieces together, healing a hyphenated word split across the break instead of leaving a space
    a, b = a.rstrip(), b.lstrip()
    if a.endswith("-"):
        return a[:-1] + b
    return (a + " " + b).strip()


def _should_merge_text(prev: Dict, blk: Dict) -> bool:
    # judges whether two text blocks are really one paragraph split by a page break, based on punctuation, links, hyphens, and lowercase starts
    a, b = prev["text"], blk["text"]
    if not a.strip() or not b.strip():
        return True
    if _is_link_only(b):
        return True
    if a.rstrip().endswith("-"):
        return True
    if not _ends_sentence(a):
        return True
    return b.lstrip()[:1].islower()


def _split_table(text: str):
    # separates a table block into its caption text and its actual grid rows, so tables can be compared and merged
    lines = text.split("\n")
    gi = next((i for i, l in enumerate(lines) if l.strip().startswith("|")), None)
    if gi is None:
        return (text.strip() or None), []
    caption = "\n".join(lines[:gi]).strip() or None
    grid = [l for l in lines[gi:] if l.strip().startswith("|")]
    return caption, grid


def _ncols(grid) -> int:
    # counts how many columns a table grid has, by splitting its first row on pipes
    return len(grid[0].strip().strip("|").split("|")) if grid else 0


def _tables_continuation(prev: Dict, blk: Dict) -> bool:
    # decides whether two tables on different pages are really one table split across the break, by checking they have the same column count
    pa, pb = prev.get("page"), blk.get("page")
    if pa is None or pb is None or pa == pb:
        return False
    _, ga = _split_table(prev["text"])
    _, gb = _split_table(blk["text"])
    return bool(ga) and bool(gb) and _ncols(ga) == _ncols(gb)


def _merge_two_tables(a_text: str, b_text: str) -> str:
    # stitches a continued table back onto the first part, dropping the second piece's repeated header row if it has one
    cap_a, ga = _split_table(a_text)
    _, gb = _split_table(b_text)
    b_body = gb[:]
    if len(b_body) >= 2 and _is_table_sep(b_body[1]):
        if ga and b_body[0].strip() == ga[0].strip():
            b_body = b_body[2:]
        else:
            b_body = [b_body[0]] + b_body[2:]
    merged = ga + b_body
    prefix = (cap_a + "\n\n") if cap_a else ""
    return prefix + "\n".join(merged)


def _merge_blocks(blocks: List[Dict]) -> List[Dict]:
    # the actual stitching pass, walking the blocks and merging paragraphs split by page breaks, links, or removed logos, plus tables continued across pages, while leaving headings as hard breaks
    out: List[Dict] = []
    for blk in blocks:
        if out:
            prev = out[-1]
            if prev["type"] == "text" and blk["type"] == "text" and _should_merge_text(prev, blk):
                prev["text"] = _join_text(prev["text"], blk["text"])
                continue
            if prev["type"] == "table" and blk["type"] == "table" and _tables_continuation(prev, blk):
                prev["text"] = _merge_two_tables(prev["text"], blk["text"])
                if not prev.get("caption") and blk.get("caption"):
                    prev["caption"] = blk["caption"]
                continue
        out.append(dict(blk))
    return out

def _load_pdf_blocks_fallback(path: Path) -> List[Dict]:
    # the plain-B plan for PDFs when pymupdf4llm isn't available, pulling raw text per page with no heading, table, or image structure
    print("  (pymupdf4llm unavailable -> basic page-based PDF parse; "
          "no heading/table/image structure)")
    blocks = []
    try:
        doc = fitz.open(str(path))
        for i, page in enumerate(doc):
            for blk in page.get_text("dict").get("blocks", []):
                if blk.get("type", 0) != 0:
                    continue
                parts = []
                for line in blk.get("lines", []):
                    for span in line.get("spans", []):
                        if span.get("text"):
                            parts.append(span["text"])
                    parts.append(" ")
                t = _clean("".join(parts))
                if t and not _is_page_artifact(t):
                    blocks.append({"type": "text", "text": t, "page": i + 1})
        doc.close()
    except Exception as e:
        print(f"  PDF error {path.name}: {e}")
        return []
    return _build_section_tree(blocks)

def load_pdf_structured(path: Path) -> List[Dict]:
    # the main PDF loader, converting to markdown with pymupdf4llm and running the whole pipeline, parse, drop logos, drop repeats, fix captions, stitch, then build the section tree, falling back to the basic parser if anything goes wrong
    if not HAVE_PYMUPDF4LLM:
        return _load_pdf_blocks_fallback(path)
    img_dir = IMG_DIR / _make_id(str(path))
    img_dir.mkdir(parents=True, exist_ok=True)
    try:
        pages = pymupdf4llm.to_markdown(
            str(path), page_chunks=True,
            write_images=DESCRIBE_IMAGES, image_path=str(img_dir),
            image_format="png", dpi=MD_DPI, margins=0, show_progress=False,
        )
    except Exception as e:
        print(f"  pymupdf4llm error {path.name}: {e}; using fallback parser")
        return _load_pdf_blocks_fallback(path)

    blocks: List[Dict] = []
    for i, pg in enumerate(pages):
        blocks.extend(_parse_markdown_blocks(pg.get("text", "") or "", i + 1))
    if not blocks:
        return []
    blocks = _filter_logo_images(blocks)
    blocks = _filter_repeated(blocks, len(pages))
    blocks = _reattach_captions(blocks)
    blocks = _merge_blocks(blocks)
    return _build_section_tree(blocks)


def load_docx_structured(path: Path) -> List[Dict]:
    # the DOCX loader, walking paragraphs and tables in true reading order so nothing gets reordered, using heading styles for structure and falling back to a paragraphs-only pass if the in-order internals aren't available
    try:
        from docx.document import Document as _DocClass
        from docx.table import Table as _Table
        from docx.text.paragraph import Paragraph as _Paragraph
        from docx.oxml.table import CT_Tbl
        from docx.oxml.text.paragraph import CT_P
    except Exception as e:
        print(f"  DOCX in-order iteration unavailable ({e}); paragraphs only")
        CT_Tbl = CT_P = None

    blocks: List[Dict] = []
    try:
        document = _docx.Document(str(path))
        if CT_P is not None:
            for child in document.element.body.iterchildren():
                if isinstance(child, CT_P):
                    para = _Paragraph(child, document)
                    t = _clean(para.text)
                    if not t:
                        continue
                    style = ((para.style.name if para.style else "") or "").lower()
                    if style.startswith("heading") or style.startswith("title"):
                        lvl = 1 if style.startswith("title") else _docx_heading_level(style)
                        blocks.append({"type": "heading", "level": lvl, "text": t, "page": None})
                    elif not _is_page_artifact(t):
                        blocks.append({"type": "text", "text": t, "page": None})
                elif isinstance(child, CT_Tbl):
                    md = _docx_table_to_md(_Table(child, document))
                    if md:
                        blocks.append({"type": "table", "text": md, "page": None})
        else:
            for para in document.paragraphs:
                t = _clean(para.text)
                if not t or _is_page_artifact(t):
                    continue
                style = ((para.style.name if para.style else "") or "").lower()
                if style.startswith("heading") or style.startswith("title"):
                    blocks.append({"type": "heading", "level": _docx_heading_level(style),
                                   "text": t, "page": None})
                else:
                    blocks.append({"type": "text", "text": t, "page": None})
    except Exception as e:
        print(f"  DOCX error {path.name}: {e}")
        return []

    blocks = _reattach_captions(blocks)
    blocks = _merge_blocks(blocks)
    sections = _build_section_tree(blocks)
    if not sections and blocks:
        sections = [{"title": path.stem, "level": 1, "page": None,
                     "paragraphs": [b for b in blocks if b["type"] != "heading"],
                     "subsections": []}]
    return sections


def load_structured(path: Path) -> List[Dict]:
    # the single entry point that picks the right loader by file extension, PDF or DOCX, and ignores anything else
    ext = path.suffix.lower()
    if ext == ".pdf":
        return load_pdf_structured(path)
    if ext == ".docx":
        return load_docx_structured(path)
    return []


SUPPORTED_EXTENSIONS = {".pdf", ".docx"}  # the only file types we ingest; everything else is skipped
print("Structure-aware loaders ready (markdown headings + tables + figures; "
      "logo filtering, page/link/logo stitching, page-split table merging)")

Structure-aware loaders ready (markdown headings + tables + figures; logo filtering, page/link/logo stitching, page-split table merging)


In [ ]:
import threading  # for the stop-event and lock used while summarising leaves in parallel
import concurrent.futures as cf  # the thread pool that runs many leaf summaries at once

def _fmt_eta(seconds: float) -> str:
    # turns a number of seconds into a tidy clock string like 1:02:03 or 4:05, dropping the hours when there aren't any
    seconds = int(max(0, seconds))
    h, rem = divmod(seconds, 3600)
    m, s = divmod(rem, 60)
    return f"{h:d}:{m:02d}:{s:02d}" if h else f"{m:d}:{s:02d}"


class Progress:
    """Single global bar over all LLM calls + per-call one-line metrics."""

    def __init__(self, total: int, desc: str = "Summarising"):
        # sets up the one progress bar for the whole build, recording the start time so we can compute rate and ETA later
        self.total = max(0, total)
        self.done = 0
        self.t0 = time.time()
        self.bar = tqdm(total=self.total, desc=desc, unit="call",
                        dynamic_ncols=True, smoothing=0.3)

    def step(self, label: str, dt: float, summary: Optional[str] = None):
        # records one finished call, ticks the bar forward, updates the ETA and rate, and optionally prints the summary it just produced
        self.done += 1
        self.bar.update(1)
        elapsed = time.time() - self.t0
        rate = self.done / elapsed if elapsed > 0 else 0.0
        remaining = (self.total - self.done) / rate if rate > 0 else 0.0
        self.bar.set_postfix_str(
            f"ETA {_fmt_eta(remaining)} | {rate:.2f}/s | last {dt:.1f}s")
        if VERBOSE_CALLS and summary is not None:
            text = re.sub(r"\s+", " ", summary).strip() or "(empty summary)"
            for line in textwrap.wrap(text, width=100):
                self.bar.write(line)
            self.bar.write("")

    def reconcile(self):
        # snaps the bar's total down to however many calls actually happened, since the upfront estimate is usually a bit high
        self.bar.total = self.done
        self.bar.refresh()

    def set_phase(self, label: str):
        # relabels the bar to show which phase we're in, figures, text, or combining
        self.bar.set_description(label)
        self.bar.refresh()

    def close(self):
        # shuts the bar down and prints the final tally of calls and wall-clock time
        self.bar.close()
        print(f"Total LLM calls: {self.done} in {_fmt_eta(time.time() - self.t0)}")


def _parse_cache_path(path: Path) -> Optional[Path]:
    # works out the cache filename for a parsed document, keyed on its modification time and size so the cache invalidates itself if the file changes
    try:
        st = path.stat()
        key = _make_id(str(path), f"{st.st_mtime_ns}:{st.st_size}")
        return PARSE_CACHE_DIR / f"{key}.json"
    except Exception:
        return None


def parse_document_cached(path: Path) -> List[Dict]:
    # parses a document but reuses a cached parse if one exists and the file hasn't changed, so we don't re-read files on every run
    cp = _parse_cache_path(path)
    if cp is not None and cp.exists() and not FORCE_REPARSE:
        try:
            return json.loads(cp.read_text(encoding="utf-8"))
        except Exception:
            pass
    sections = load_structured(path)
    if cp is not None:
        try:
            cp.write_text(json.dumps(sections, ensure_ascii=False), encoding="utf-8")
        except Exception:
            pass
    return sections


def clear_parse_cache() -> int:
    # wipes the parsed-document cache and reports how many files it removed, which is what FORCE_REPARSE triggers
    n = 0
    for p in PARSE_CACHE_DIR.glob("*.json"):
        p.unlink(); n += 1
    print(f"Cleared {n} cached parse(s) from {PARSE_CACHE_DIR}")
    return n


def _walk_sections(sections: List[Dict], parent_extra: Optional[str] = None):
    # walks every section and subsection yielding an id suffix for each, built the exact same way the builder builds ids so the cache lines up perfectly
    for i, sec in enumerate(sections):
        extra = f"s{i}" if parent_extra is None else f"{parent_extra}.{i}"
        yield extra, sec
        yield from _walk_sections(sec["subsections"], extra)


def _iter_docs(folder: Path):
    # yields every supported document under a folder, going depth-first with subfolders before files and skipping hidden entries, so files come out in a stable order
    try:
        entries = sorted(folder.iterdir(), key=lambda p: (p.is_file(), p.name.lower()))
    except Exception:
        return
    for e in entries:
        if e.name.startswith("."):
            continue
        if e.is_dir():
            yield from _iter_docs(e)
    for e in entries:
        if e.name.startswith("."):
            continue
        if e.is_file() and e.suffix.lower() in SUPPORTED_EXTENSIONS:
            yield e


def _leaf_name(path: Path, sec_title: str, kind: str, pi: int) -> str:
    # builds a readable name for a leaf unit like "report.pdf - Methods - table 3"
    word = {"text": "para", "table": "table", "image": "figure"}.get(kind, "para")
    return f"{path.name} - {sec_title} - {word} {pi}"


def _leaf_meta(path: Path, sec_title: str, para: Dict, pi: int, kind: str) -> Dict:
    # assembles the metadata dict for a leaf, source file, section, kind, and optionally page, caption, and image path when present
    meta = {"source_file": str(path), "section": sec_title, "unit_index": pi,
            "file_type": path.suffix.lower(), "kind": kind}
    if para.get("page") is not None:
        meta["page"] = para["page"]
    if para.get("caption"):
        meta["caption"] = para["caption"]
    if kind == "image" and para.get("path"):
        meta["image_path"] = para["path"]
    return meta


def plan_corpus(root: Path):
    # the planning pass that reads every document, figures out which leaves still need summarising versus which are cached or short enough to store verbatim, and estimates how many combine calls are coming, so the progress bar has a total to work with
    docs = list(_iter_docs(root))
    print(f"Discovered {len(docs)} document(s) under {root}")
    parsed: Dict[str, List[Dict]] = {}
    leaf_jobs: List[Dict] = []
    n_sections = 0
    n_prefilled = 0

    for path in tqdm(docs, desc="Parsing", unit="doc", dynamic_ncols=True):
        sections = parse_document_cached(path)
        parsed[str(path)] = sections
        for extra, sec in _walk_sections(sections):
            n_sections += 1
            for pi, para in enumerate(sec["paragraphs"]):
                cid = _make_id(str(path), f"{extra}p{pi}")
                if _load_cached_node(cid) is not None:
                    continue
                kind = para.get("type", "text")
                if kind == "text" and len(_words(para["text"])) <= SKIP_SUMMARY_WORDS:
                    # too short to summarise meaningfully -> store text as its own summary
                    node = TreeNode(node_id=cid, node_type="chunk",
                                    name=_leaf_name(path, sec["title"], kind, pi), path="",
                                    summary=para["text"], content=para["text"],
                                    metadata=_leaf_meta(path, sec["title"], para, pi, kind))
                    _save_cached_node(node)
                    n_prefilled += 1
                    continue
                leaf_jobs.append({"path": str(path), "sec_title": sec["title"],
                                  "extra": extra, "pi": pi, "para": para,
                                  "cid": cid, "kind": kind})

    folders = {str(Path(p).resolve().parent) for p in parsed}
    folders.add(str(root.resolve()))
    est_combines = n_sections + len(docs) + len(folders) + 1
    print(f"  sections: {n_sections} | LLM leaf calls: {len(leaf_jobs)} | "
          f"pre-filled short units: {n_prefilled} | est. combine calls: {est_combines}")
    return docs, parsed, leaf_jobs, est_combines


def _summarise_leaf(job: Dict):
    # does the actual work for one leaf, describing an image, summarising a table, or summarising text, dropping it if a captionless image turns out to be a logo, then caches the result and reports back how it went
    path = Path(job["path"]); para = job["para"]; kind = job["kind"]
    sec_title = job["sec_title"]; cid = job["cid"]; pi = job["pi"]
    t0 = time.time()
    caption = para.get("caption", "")
    meta = _leaf_meta(path, sec_title, para, pi, kind)
    name = _leaf_name(path, sec_title, kind, pi)
    label = f"{kind}: {path.name} / {sec_title}"
    try:
        if kind == "image":
            desc = _llm_describe_image(para.get("path", ""), caption_hint=caption)
            if DROP_LOGO_BY_DESC and not caption and _looks_like_logo(desc):
                meta["skipped"] = "logo"
                node = TreeNode(node_id=cid, node_type="chunk", name=name, path="",
                                summary="", content="", metadata=meta)
                _save_cached_node(node)
                return ("skip", time.time() - t0, label + " [logo]", f"(skipped logo) {desc}")
            content = ("[FIGURE] " + (caption + " — " if caption else "") + desc).strip()
            summary = desc
        elif kind == "table":
            content = para["text"]
            summary = _llm_summarise(content, context_hint=f"{path.name}, {sec_title} (Markdown table)")
        else:
            content = para["text"]
            summary = _llm_summarise(content, context_hint=f"{path.name}, {sec_title}")
        node = TreeNode(node_id=cid, node_type="chunk", name=name, path="",
                        summary=summary, content=content, metadata=meta)
        _save_cached_node(node)
        return ("ok", time.time() - t0, label, summary)
    except Exception as e:
        return ("err", time.time() - t0, f"{label} -> {type(e).__name__}: {e}",
                f"(error: {e})")


def _wait_until_ready():
    """Block until the Ollama server is reachable AND the required model(s) are
    present. Does NOT raise — if the tunnel is down or a model is still being
    pulled, it just keeps checking. (Interrupt the cell if you need to stop.)"""
    # politely waits for the server and the models it needs to be there before the build starts, looping forever rather than crashing if the tunnel is down or a model is still pulling
    need = [SUMMARY_MODEL] + ([VISION_MODEL] if DESCRIBE_IMAGES else [])
    announced = False
    while True:
        reason = None
        try:
            names = _model_names(client.list())
            missing = [m for m in need if not any(m in n for n in names)]
            if not missing:
                if announced:
                    print("Ollama reconnected.")
                print(f"Ollama OK — models present: {', '.join(need)}")
                return
            reason = f"model(s) not loaded yet: {missing} (pull them in another terminal)"
        except Exception as e:
            reason = f"server unreachable ({type(e).__name__}: {e})"
        if not announced:
            print(f"Waiting for Ollama — {reason}. Re-checking every {READY_POLL}s; "
                  "this will resume automatically, it won't stop.")
            announced = True
        time.sleep(READY_POLL)


def _warm(model: str):
    """Load a model into memory once (with keep_alive) so the first real call in a
    phase isn't paying the cold-load cost, and so we don't swap models mid-phase."""
    # nudges a model into memory with a tiny throwaway call before a phase starts, so the first real call isn't stuck waiting on a cold load
    try:
        client.chat(model=model, messages=[{"role": "user", "content": "ok"}],
                    options={"num_predict": 1}, keep_alive=KEEP_ALIVE)
    except Exception:
        pass


def fill_leaves(leaf_jobs: List[Dict], progress: Progress, workers: Optional[int] = None):
    # runs a batch of leaf jobs across a thread pool for real parallelism, and if any one of them errors it stops the whole batch and raises, so a bad call doesn't get silently cached and you can just re-run to resume
    if not leaf_jobs:
        return
    stop = threading.Event()

    def run(job):
        # the per-thread wrapper that bails out early if another worker has already hit an error
        if stop.is_set():
            return ("cancelled", 0.0, "", None)
        return _summarise_leaf(job)

    with cf.ThreadPoolExecutor(max_workers=workers or NUM_WORKERS) as ex:
        futs = [ex.submit(run, j) for j in leaf_jobs]
        for fut in cf.as_completed(futs):
            try:
                status, dt, label, summary = fut.result()
            except Exception as e:
                status, dt, label, summary = "err", 0.0, f"worker crashed: {e}", str(e)
            if status == "err":
                stop.set()
                for f in futs:
                    f.cancel()
                raise RuntimeError(
                    f"LLM call failed ({label}). Build stopped; the failed unit was "
                    "NOT cached. Fix the connection/model and re-run — completed "
                    "summaries are cached, so it resumes where it left off.")
            if status == "cancelled":
                continue
            progress.step(label, dt, summary)


def _build_section_node(path: Path, sec: Dict, extra: str, progress: Progress) -> Optional[TreeNode]:
    # builds one section node by gathering its already-summarised leaves and its subsections, then combining their summaries into one, reusing the cache if this section was already done
    sid = _make_id(str(path), extra)
    cached = _load_cached_node(sid)
    if cached is not None and cached.node_type == "section":
        return cached

    children: List[TreeNode] = []
    for pi, _para in enumerate(sec["paragraphs"]):
        leaf = _load_cached_node(_make_id(str(path), f"{extra}p{pi}"))
        if leaf is None or leaf.metadata.get("skipped"):
            continue
        children.append(leaf)
    for ci, sub in enumerate(sec["subsections"]):
        sn = _build_section_node(path, sub, f"{extra}.{ci}", progress)
        if sn is not None:
            children.append(sn)
    if not children:
        return None

    t0 = time.time()
    summary = _llm_combine_summaries([c.summary for c in children], label=sec["title"])
    progress.step(f"section: {path.name} / {sec['title']}", time.time() - t0, summary)
    node = TreeNode(node_id=sid, node_type="section",
                    name=f"{path.name} - {sec['title']}", path="",
                    summary=summary, children=children,
                    metadata={"source_file": str(path), "section": sec["title"],
                              "level": sec.get("level"), "page": sec.get("page")})
    _save_cached_node(node)
    return node


def _build_document_node(path: Path, sections: List[Dict], progress: Progress) -> Optional[TreeNode]:
    # builds one document node by building each of its sections and combining their summaries into a document-level summary, again reusing the cache when possible
    did = _make_id(str(path))
    cached = _load_cached_node(did)
    if cached is not None and cached.node_type == "document":
        return cached
    sec_nodes = []
    for i, sec in enumerate(sections):
        sn = _build_section_node(path, sec, f"s{i}", progress)
        if sn is not None:
            sec_nodes.append(sn)
    if not sec_nodes:
        return None
    t0 = time.time()
    summary = _llm_combine_summaries([n.summary for n in sec_nodes], label=path.name)
    progress.step(f"document: {path.name}", time.time() - t0, summary)
    node = TreeNode(node_id=did, node_type="document", name=path.name, path=str(path),
                    summary=summary, children=sec_nodes,
                    metadata={"file_type": path.suffix.lower(), "num_sections": len(sec_nodes)})
    _save_cached_node(node)
    return node


def _build_folder_node(folder: Path, parsed: Dict[str, List[Dict]], progress: Progress,
                       is_root: bool = False) -> Optional[TreeNode]:
    # builds one folder node by recursing into subfolders first and then documents, combining all their summaries, and it's this same function that produces the top-level root node when is_root is set
    fid = _make_id(str(folder))
    cached = _load_cached_node(fid)
    if cached is not None and cached.node_type in ("folder", "root"):
        return cached
    try:
        entries = sorted(folder.iterdir(), key=lambda p: (p.is_file(), p.name.lower()))
    except Exception:
        return None

    children: List[TreeNode] = []
    for e in entries:
        if e.name.startswith(".") or not e.is_dir():
            continue
        n = _build_folder_node(e, parsed, progress)
        if n is not None:
            children.append(n)
    for e in entries:
        if e.name.startswith(".") or not e.is_file():
            continue
        if e.suffix.lower() not in SUPPORTED_EXTENSIONS:
            continue
        sections = parsed.get(str(e)) or parse_document_cached(e)
        dn = _build_document_node(e, sections, progress)
        if dn is not None:
            children.append(dn)
    if not children:
        return None

    label = folder.name or str(folder)
    t0 = time.time()
    summary = _llm_combine_summaries([c.summary for c in children], label=label)
    progress.step(f"{'root' if is_root else 'folder'}: {label}", time.time() - t0, summary)
    node = TreeNode(node_id=fid, node_type=("root" if is_root else "folder"),
                    name=label, path=str(folder), summary=summary, children=children,
                    metadata={"num_children": len(children)})
    _save_cached_node(node)
    return node


def build_corpus(root: Path) -> Optional[TreeNode]:
    # the top-level orchestrator, it waits for the server, plans the work, then runs the three phases in order, all figures first, then all text, then all the combines, so each model loads once instead of swapping constantly, and finally saves nothing here but returns the finished root node
    if not root.exists():
        print(f"DOCS_ROOT does not exist: {root.resolve()}")
        return None
    _wait_until_ready()                   # waits here until the server/model is reachable
    t_start = time.time()
    docs, parsed, leaf_jobs, est_combines = plan_corpus(root)
    if not docs:
        print(f"No supported (.pdf/.docx) documents under {root.resolve()}")
        return None

    total = len(leaf_jobs) + est_combines
    progress = Progress(total, desc="Summarising corpus")

    image_jobs = [j for j in leaf_jobs if j["kind"] == "image"]
    text_jobs = [j for j in leaf_jobs if j["kind"] != "image"]

    try:
        if image_jobs:
            progress.set_phase(f"1/3 figures [{VISION_MODEL}]")
            _warm(VISION_MODEL)
            fill_leaves(image_jobs, progress, workers=VISION_WORKERS)

        if text_jobs:
            progress.set_phase(f"2/3 text [{SUMMARY_MODEL}]")
            _warm(SUMMARY_MODEL)
            fill_leaves(text_jobs, progress, workers=NUM_WORKERS)

        progress.set_phase(f"3/3 combining [{SUMMARY_MODEL}]")
        _warm(SUMMARY_MODEL)
        root_node = _build_folder_node(root, parsed, progress, is_root=True)
    except Exception:
        progress.close()
        raise
    progress.reconcile()
    progress.close()
    print(f"Corpus built in {_fmt_eta(time.time() - t_start)}")
    return root_node

def print_tree_summaries(node: TreeNode, depth: int = 0, max_depth: int = 99,
                         show_leaves: bool = False) -> None:
    # prints the finished tree as an indented outline of summaries, stopping at a max depth and skipping the leaf units unless you ask to see them
    if depth > max_depth:
        return
    kind = node.metadata.get("kind")
    base = {"root": "ROOT", "folder": "FOLDER", "document": "DOCUMENT",
            "section": "SECTION", "chunk": "UNIT"}.get(node.node_type, node.node_type.upper())
    if node.node_type == "chunk":
        base = {"text": "PARAGRAPH", "table": "TABLE", "image": "FIGURE"}.get(kind, "PARAGRAPH")
        if not show_leaves:
            return
    pad = "  " * depth
    loc = f" (page {node.metadata['page']})" if node.metadata.get("page") is not None else ""
    n_leaves = node.count_leaves()
    extra = f"  [{n_leaves} units]" if node.node_type in ("root", "folder", "document") else ""
    print(f"{pad}{base}: {node.name}{loc}{extra}")
    summ = re.sub(r"\s+", " ", node.summary or "").strip() or "(no summary)"
    for line in textwrap.wrap(summ, width=88):
        print(f"{pad}    {line}")
    print("")
    for child in node.children:
        print_tree_summaries(child, depth + 1, max_depth, show_leaves)


print("Corpus builder ready (parallel leaves + bottom-up folder nesting + progress/ETA)")

Corpus builder ready (parallel leaves + bottom-up folder nesting + progress/ETA)


In [ ]:
def print_tree_full(node: TreeNode, depth: int = 0) -> None:
    # the deep-dive printer for spot-checking one node, showing not just summaries but the raw text, tables, and descriptions underneath, which is why you point it at a single document rather than the whole corpus
    kind = node.metadata.get("kind")
    base = {"root": "ROOT", "folder": "FOLDER", "document": "DOCUMENT",
            "section": "SECTION", "chunk": "PARAGRAPH"}.get(node.node_type, node.node_type.upper())
    if node.node_type == "chunk":
        base = {"text": "PARAGRAPH", "table": "TABLE", "image": "FIGURE"}.get(kind, "PARAGRAPH")
    pad = "  " * depth
    loc = f" (page {node.metadata['page']})" if node.metadata.get("page") is not None else ""
    print(f"{pad}{base}: {node.name}{loc}")
    if node.metadata.get("caption"):
        print(f"{pad}  CAPTION: {node.metadata['caption']}")
    summ = re.sub(r"\s+", " ", node.summary or "").strip() or "(no summary)"
    print(f"{pad}  {'DESCRIPTION' if kind == 'image' else 'SUMMARY'}:")
    for line in textwrap.wrap(summ, width=92):
        print(f"{pad}    {line}")
    if node.content:
        if kind == "table":
            print(f"{pad}  ORIGINAL TABLE:")
            for line in node.content.split("\n"):
                print(f"{pad}    {line}")
        else:
            orig = re.sub(r"\s+", " ", node.content).strip()
            print(f"{pad}  ORIGINAL {'CONTENT' if kind == 'image' else 'TEXT'}:")
            for line in textwrap.wrap(orig, width=92):
                print(f"{pad}    {line}")
    print("")
    for child in node.children:
        print_tree_full(child, depth + 1)


def find_node(node: TreeNode, name_substr: str) -> Optional[TreeNode]:
    # hunts through the tree for the first node whose name contains a given substring, like a filename, so you can grab a document to feed into print_tree_full
    if name_substr.lower() in node.name.lower():
        return node
    for c in node.children:
        hit = find_node(c, name_substr)
        if hit:
            return hit
    return None


print("Drill-down printer ready (use print_tree_full(find_node(root, 'somefile.pdf')))")

Drill-down printer ready (use print_tree_full(find_node(root, 'somefile.pdf')))


In [ ]:
if FORCE_REBUILD:
    clear_node_cache()
if FORCE_REPARSE:
    clear_parse_cache()
repair_cache()
cache_status()
print("=" * 72)

root_node = build_corpus(DOCS_ROOT)

if root_node:
    print("")
    print("#" * 72)
    print(f"CORPUS ROOT - {root_node.name}")
    print(f"  {root_node.count_nodes()} nodes | {root_node.count_leaves()} leaf units")
    print("#" * 72)
    print("")
    print_tree_summaries(root_node, max_depth=CORPUS_PRINT_DEPTH,
                         show_leaves=SHOW_LEAVES_IN_TREE)

    with open(TREE_FILE, "w", encoding="utf-8") as f:
        json.dump(root_node.to_dict(), f, ensure_ascii=False, indent=2)
    print(f"(Saved corpus tree JSON to {TREE_FILE})")

    print("\n" + "=" * 72)
    print("ROOT SUMMARY")
    print("=" * 72)
    print(re.sub(r"\s+", " ", root_node.summary).strip())

Cache clean: no failed (could-not-connect) nodes found
Per-node cache @ tree_cache/nodes:
  chunk     : 97328
  document  : 2161
  folder    : 306
  section   : 25699
  total size: 809249.4 KB
Ollama OK — models present: gpt-oss:120b, gemma3:27b
Discovered 2852 document(s) under folders


Parsing: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2852/2852 [00:11<00:00, 258.79doc/s]


  sections: 40591 | LLM leaf calls: 0 | pre-filled short units: 0 | est. combine calls: 43818


3/3 combining [gpt-oss:120b]:   0%|                                                     | 1/43818 [00:04<53:31:56,  4.40s/call, ETA 53:34:05 | 0.23/s | last 2.9s]

The section defines the mandatory data elements that must appear on every laboratory test
report—paper or electronic—and be stored in the laboratory information system. Required items
include the testing laboratory’s name and address, patient name and identifier, ordering physician
(or legally authorized person), specimen collection date/time, report release date/time, specimen
source, test results with units, reference intervals, and any specimen conditions affecting
adequacy. All elements must be readily accessible to clinicians; electronic reports may distribute
them across screens but must allow retrieval. Referral laboratories must transmit results back to
the referring lab unless exempted (e.g., drug‑of‑abuse testing). The ordering physician must be
identifiable via audit trails, even in rotating‑physician settings. Controlled distribution of
reference‑interval tables is allowed. References cite CMS regulations, CLSI EP28‑A3C, and
reference‑value literature.



...

The Technical SOPs and Worksheets folder is a comprehensive, end‑to‑end reference for all laboratory
activities in OICR Genomics and the Tissue Portal. It standardises sample receipt, accessioning, and
biobanking (blood fractionation, plasma, buffy‑coat, FFPE, fresh‑frozen, LCM); nucleic‑acid
extraction (cfDNA, gDNA, RNA) using manual kits, KingFisher, and automated platforms;
library‑preparation workflows for whole‑genome, whole‑transcriptome, targeted‑amplicon (REVOLVE) and
plasma‑WGS assays on Illumina NovaSeq, NextSeq and MiSeq; instrument‑specific SOPs (Covaris,
Hamilton STAR, Perkin Elmer Sciclone, NovaSeq X Plus, NextSeq 2000, Illumina PhiX, Qubit,
TapeStation, Fragment Analyzer, Dimsum QC dashboard); data‑processing pipelines, variant‑calling,
and clinical‑report generation with geneticist sign‑off; LIMS (MISO) usage, library custody,
aliquoting, inventory, and sample‑transfer procedures; safety, PPE, risk‑assessment, and
waste‑disposal guidelines; and ready‑to‑use Markdown wor

3/3 combining [gpt-oss:120b]:  15%|███████▏                                        | 6530/43818 [6:03:59<60:24:37,  5.83s/call, ETA 34:38:30 | 0.30/s | last 5.4s]

The folders collection is the complete quality‑management repository for OICR Genomics. It houses
validated assay documentation (CHARM ctDNA, plasma‑WGS, REVOLVE, WGS/WTS), detailed extraction and
bio‑informatics performance metrics, and CAP‑mandated verification procedures. Centralized CAPA logs
(2019‑2023) record investigations, corrective actions and preventive plans for laboratory,
informatics and compliance incidents. Change‑request and Improvement‑Action‑Plan files (2021‑2025)
drive SOP, assay and workflow updates, automation projects, and QC enhancements. The Lab Quality
Documents, Management, and Planned‑Deviation folders provide accreditation records (ISO 15189, CAP,
CLIA), audit reports, KPI reviews, risk matrices, biosafety permits and privacy‑breach
investigations, ensuring governance and regulatory adherence. Comprehensive SOP and worksheet suites
cover quality (validation, KPI, document control), safety (chemical, biosafety, ergonomics, WHMIS),
and technical operations (s

3/3 combining [gpt-oss:120b]: 100%|████████████████████████████████████████████████████| 6530/6530 [6:04:02<00:00,  3.34s/call, ETA 34:38:30 | 0.30/s | last 5.4s]


Total LLM calls: 6530 in 6:04:02
Corpus built in 6:04:13

########################################################################
CORPUS ROOT - folders
  130154 nodes | 95458 leaf units
########################################################################

ROOT: folders  [95458 units]
    The folders collection is the complete quality‑management repository for OICR Genomics.
    It houses validated assay documentation (CHARM ctDNA, plasma‑WGS, REVOLVE, WGS/WTS),
    detailed extraction and bio‑informatics performance metrics, and CAP‑mandated
    verification procedures. Centralized CAPA logs (2019‑2023) record investigations,
    corrective actions and preventive plans for laboratory, informatics and compliance
    incidents. Change‑request and Improvement‑Action‑Plan files (2021‑2025) drive SOP, assay
    and workflow updates, automation projects, and QC enhancements. The Lab Quality
    Documents, Management, and Planned‑Deviation folders provide accreditation records (ISO
    1